# Process Processed Data: Static XAS Good-Shot Selection

This notebook starts from `run{RUN_NO}_static_xas.h5` and writes `run{RUN_NO}_good_static_xas.h5` for hall GMD or `run{RUN_NO}_good_static_xas_tunnel.h5` for tunnel GMD.

The important diagnostic here is the VLS/GMD correlation check before and after the weak-VLS peak filter:

- nominal axis: section/wavelength energy groups,
- converted axis: VLS center proxy-pixel bins, then converted to eV.

Most logic lives in `analysis/scripts/static_xas_process_good.py`; this notebook stays intentionally thin.

Energy conversion uses the fixed spectrometer geometry and the anchor
**524 pixel = 284.2 eV**, matching the previous static-XAS notebooks.
The section-energy comparison is a diagnostic, not a fitted calibration.
The default peak center is a local quadratic fit (`smooth_peak`).

Existing good H5 files are unchanged until the final save cell is run.
To update converted energies, rerun this notebook for the required runs and
GMD sources. Reference-normalized XAS needs both sample and reference good
files rebuilt with the same physical-map settings.


In [ ]:
from __future__ import annotations

import importlib
import os
import sys
import tempfile
from pathlib import Path

mpl_cache = Path(tempfile.gettempdir()) / "glycine26-matplotlib-cache"
mpl_cache.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))

import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 160})


def find_repo_root(start: Path) -> Path:
    for parent in (start.resolve(), *start.resolve().parents):
        marker = parent / "analysis" / "scripts" / "static_xas_process_good.py"
        if marker.exists():
            return parent
    raise RuntimeError("Could not find repo root from notebook path.")


REPO_ROOT = find_repo_root(Path.cwd())
SCRIPTS_DIR = REPO_ROOT / "analysis" / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import config as path_config
import static_xas_process_good as pg
pg = importlib.reload(pg)

RUN_NO = 58794
INPUT_H5 = Path(path_config.COMBINED_DIR).parent / "xas_static" / f"run{RUN_NO}_static_xas.h5"
PROCESS_GMD_SOURCE = "hall"  # "hall" or "tunnel"
GOOD_SUFFIX = "_good_static_xas_tunnel.h5" if PROCESS_GMD_SOURCE == "tunnel" else "_good_static_xas.h5"
OUTPUT_H5 = INPUT_H5.with_name(f"run{RUN_NO}{GOOD_SUFFIX}")

# Fixed spectrometer map from the previous static-XAS notebooks.
# Section energies are diagnostic coordinates; they do not fit this map.
PIXEL_MAP_CONFIG = {
    "source_to_grating_m": 0.85,
    "grating_to_screen_m": 0.5429,
    "incident_angle_deg": 86.78,
    "groove_density_lines_per_mm": 1200.0,
    "diffraction_order": 1,
    "magnification": 3.0,
    "pixel_pitch_m": 50e-6,
    "axis_sign": 1,
}
PIXEL_MAP_CALIBRATION = {"pixel": 524.0, "energy_eV": 284.2}

CONFIG = pg.ProcessGoodConfig(
    # Input/output and GMD source (also determines intercept corrections).
    input_h5=INPUT_H5,
    output_h5=OUTPUT_H5,
    gmd_source=PROCESS_GMD_SOURCE,
    chunk_size=20_000,

    # Peak center: smooth_peak = quadratic vertex; invalid fits fall back to com.
    # Other choices: com (legacy positive COM), peak (argmax),
    # gaussian_logfit (log-quadratic), local_com (local-baseline-subtracted COM).
    center_method="smooth_peak",
    center_half_window_px=5,          # strongest peak +/- 5 pixels; peak ignores this
    center_baseline_percentile=20.0,  # ONLY local_com; separate from side-background QC

    # Absolute Gotthard pixel -> eV: fixed physical map, no section polynomial fit.
    pixel_map_config=PIXEL_MAP_CONFIG,
    pixel_map_calibration=PIXEL_MAP_CALIBRATION,
    inverse_map_range_ev=(260.0, 290.0),  # out-of-range pixels raise; no extrapolation
    inverse_map_samples=10_000,

    # Shot QC and peak significance (independent of the center method).
    use_ml_qc_ok=True,
    peak_snr_threshold=5.0,            # retain prominence / background sigma > threshold
    prominence_inner_px=5,            # exclude peak core from side-background windows
    prominence_outer_px=20,           # outer extent of side-background windows
    prominence_baseline_percentile=20.0,  # percentile on EACH side; use larger baseline
    sigma_half_window_px=5,           # median raw noise sigma near the strongest peak
    min_side_pixels=5,                # minimum available pixels on each side

    # Energy grouping for diagnostics and saved converted proxy bins.
    energy_group_decimals=3,          # rounding of section energy for nominal groups
    proxy_pixel_bin_width=1,          # width of peak-center bins in Gotthard pixels
)

# Single-energy visual correlation demo.
CORRELATION_COLORMAP_ENERGY_EV = 273.0
CORRELATION_COLORMAP_AXIS = "nominal"  # "nominal" or "converted"
CORRELATION_COLORMAP_STAGE = "after_peak_filter"  # "before_peak_filter" or "after_peak_filter"
CORRELATION_COLORMAP_GRIDSIZE = 75

INPUT_H5, OUTPUT_H5


In [ ]:
result = pg.compute_processed_qc(CONFIG)
pg.print_processed_qc_summary(result)

cols = [
    "section", "section_energy_eV", "recorded", "basic", "center_valid",
    "peak_keep", "basic_fraction", "peak_fraction_of_basic", "center_pixel",
]
display(result.counts_df[cols].style.format({
    "section_energy_eV": "{:.3f}",
    "basic_fraction": "{:.3f}",
    "peak_fraction_of_basic": "{:.3f}",
    "center_pixel": "{:.2f}",
}))

print("Nominal R² before/after peak filter")
display(result.fit_nominal_before.dataframe()[["energy_eV", "n", "r2"]].rename(columns={"n": "n_before", "r2": "r2_before"}).merge(
    result.fit_nominal_after.dataframe()[["energy_eV", "n", "r2"]].rename(columns={"n": "n_after", "r2": "r2_after"}),
    on="energy_eV",
).style.format({"energy_eV": "{:.3f}", "r2_before": "{:.3f}", "r2_after": "{:.3f}"}))


In [ ]:
fig, axes = pg.plot_peak_filter_corr_summary(result)
plt.show()

fig_cal, axes_cal = pg.plot_energy_calibration_summary(result)
plt.show()

fig_corr, ax_corr = pg.plot_single_energy_correlation_colormap(
    result,
    energy_eV=CORRELATION_COLORMAP_ENERGY_EV,
    axis=CORRELATION_COLORMAP_AXIS,
    stage=CORRELATION_COLORMAP_STAGE,
    gridsize=CORRELATION_COLORMAP_GRIDSIZE,
)
plt.show()


In [ ]:
saved = pg.save_good_static_xas(result)
print(f"saved good H5: {saved}")
